# 08 · Register pressure from PTX vs ptxas

For every compiled config, the saved PTX is parsed, split into basic blocks and run through a
backward liveness analysis. The peak number of values alive at once is an estimate of register
pressure made before ptxas runs. It is compared here with the register count ptxas actually chose.

In [ ]:
# Setup: mount Drive, make sure the pinned torch/triton are installed, import archtune.
import importlib.metadata as md, os, subprocess, sys
PROJECT = "/content/drive/MyDrive/archtune"
NEED_GPU_STACK = False

if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

def _ver(pkg):
    try:
        return md.version(pkg).split("+")[0]
    except md.PackageNotFoundError:
        return None

if NEED_GPU_STACK and (_ver("torch"), _ver("triton")) != ("2.6.0", "3.2.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "triton==3.2.0"], check=True)
    if "torch" in sys.modules:
        raise SystemExit("torch was already imported: restart the session and run this cell again")

from archtune import config, gpu, model, compiler, run
ENV = gpu.capture()
MINI = config.mini_mode()
RESULTS = os.environ.get("ARCHTUNE_RESULTS") or os.path.join(PROJECT, "results_mini" if MINI else "results")
print(ENV["gpu_name"], "| torch", ENV["torch_version"], "| triton", ENV["triton_version"], "|", RESULTS)
if NEED_GPU_STACK and not gpu.pinned_ok(ENV):
    raise SystemExit("wrong torch/triton loaded: restart the session and run this cell again")

In [ ]:
import os, pandas as pd
GPU = ENV["gpu_tag"] or "rtx3050_sm86"   # results folder to analyse
SW = run.load_sweep(RESULTS, GPU)
rows = []
for r in SW[(SW.kernel == "matmul") & SW.artifact_path.notna()].drop_duplicates("cfg_hash").itertuples():
    ptx = gpu.load_artifact(r.artifact_path, "ptx")
    if ptx:
        rows.append({"cfg_hash": r.cfg_hash, "n_regs": r.n_regs, "n_spills": r.n_spills, **compiler.peak_pressure(ptx)})
lv = pd.DataFrame(rows); lv.to_parquet(os.path.join(RESULTS, GPU, "ptx_liveness.parquet"))
run.fig_liveness(lv, f"PTX liveness vs ptxas — {GPU}", os.path.join(RESULTS, "figures", f"fig_liveness_{GPU}.png"))
lv.describe()